# Docling PDF 파싱 성능 검증 (Parsing Evaluation / PoC)

이 노트북은 PDF → Markdown 변환 테스트가 **아니라**, Docling이 PDF의 문서 구조를
얼마나 정확히 복원하는지 검증하기 위한 평가 노트북이다. 실행 후 다음 세 질문에 답할 수 있어야 한다.

| # | 평가 영역 | 핵심 질문 |
|---|---|---|
| 1 | **Layout Analysis** | Text / Heading / Table / Picture / List 등 영역을 올바른 문서 요소로 구분하는가? |
| 2 | **Text Hierarchy** | Heading → Subheading → Paragraph 계층과 문서 순서를 보존하는가? |
| 3 | **Table Structure** | 표의 Row / Column / Header / Cell / 병합 구조를 복원하는가? |

**구성**

1. 환경 및 라이브러리 설정 → 2. PDF·경로 설정 → 3. Docling 문서 변환 → 3-1. 파싱 결함 후처리 →
3-2. 검수 시트(정답지 없는 문서) → 4. Layout Analysis → 5. Text Hierarchy → 6. Table Structure →
7. 전체 요약 → 8. Markdown 저장

**실행 환경 주의 (이 머신 기준)**

- Docling은 base 파이썬이 아니라 conda env `docling`에 설치되어 있다
  (`C:\Users\young\miniconda3\envs\docling\python.exe`, docling 2.119.0 / Python 3.12).
  해당 env의 커널로 실행할 것.
- 실행 산출물은 모두 `docling_eval/output/` 아래에만 생성된다.

---
## 1. 환경 및 라이브러리 설정

`TORCHDYNAMO_DISABLE=1`을 **torch/docling import 이전에** 설정한다. 레이아웃 모델이 `torch.compile`을
사용하는데, MSVC(`cl.exe`)·triton이 없는 환경에서는 전 페이지가 `Compiler: cl is not found`로 실패해
변환 전체가 `status=failure`가 된다.

In [1]:
import os

# torch import 이전에 반드시 설정 (아래 설명 참조)
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

import json
import re
import sys
from importlib.metadata import version as pkg_version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import DocItemLabel, ListItem, TableItem

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 200)

print(f"python       : {sys.version.split()[0]}")
for pkg in ("docling", "docling-core", "pandas"):
    print(f"{pkg:13}: {pkg_version(pkg)}")

python       : 3.12.13
docling      : 2.119.0
docling-core : 2.91.0
pandas       : 3.0.5


---
## 2. PDF 파일 및 경로 설정

- `BASE_DIR`는 노트북을 **레포 루트에서 열든 `docling_eval/` 안에서 열든** 항상
  `.../docling_eval` 하나를 가리키도록 현재 작업 디렉터리를 보고 결정한다(`docling_eval/docling_eval` 방지).
- **검증할 PDF를 바꾸려면 `PDF_PATH` 한 줄만 수정**한다. 대상은 두 갈래다 —
  사내 규정 `tiger_inc/pdf/`, 법령 `tiger_inc/law/`. 아래 후보 목록에 둘 다 찍힌다.
- 법령은 조판·생성기가 달라 §3-1에서 **적용할 교정 단계가 자동으로 갈린다**(그 셀 설명 참조).


In [2]:
CWD = Path.cwd().resolve()
BASE_DIR = CWD if CWD.name == "docling_eval" else CWD / "docling_eval"
BASE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = BASE_DIR / "output"
LAYOUT_DIR = OUTPUT_DIR / "layout"
TABLE_DIR = OUTPUT_DIR / "tables"
for _d in (OUTPUT_DIR, LAYOUT_DIR, TABLE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

REPO_ROOT = BASE_DIR.parent
DOC_DIRS = [REPO_ROOT / "tiger_inc" / "pdf", REPO_ROOT / "tiger_inc" / "law"]

# ▼▼▼ 검증 대상 PDF — 다른 문서를 보려면 이 줄만 바꾼다 (법령은 .../tiger_inc/law/) ▼▼▼
PDF_PATH = REPO_ROOT / "tiger_inc" / "pdf" / "법인카드_사용규정.pdf"
# ▲▲▲

if not PDF_PATH.exists():
    _fallback = BASE_DIR / "sample.pdf"
    if _fallback.exists():
        PDF_PATH = _fallback
    else:
        raise FileNotFoundError(
            "PDF를 찾을 수 없습니다.\n"
            f"  1순위(PDF_PATH): {PDF_PATH}\n"
            f"  2순위(fallback): {_fallback}\n"
            "→ 위 셀의 PDF_PATH 를 실제 파일 경로로 바꾸거나, sample.pdf 를 docling_eval/ 아래에 두세요."
        )

print(f"BASE_DIR   : {BASE_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"PDF_PATH   : {PDF_PATH}")
print(f"size       : {PDF_PATH.stat().st_size / 1024:.1f} KB")

# 참고: 바꿔가며 볼 수 있는 후보들 (규정 / 법령)
for _dir in DOC_DIRS:
    _candidates = sorted(_dir.glob("*.pdf"))
    if _candidates:
        print(f"\n[{_dir.name}] 선택 가능한 PDF")
        for _c in _candidates:
            print("  -", _c.relative_to(REPO_ROOT).as_posix())


BASE_DIR   : D:\project\SKN29-FINAL-1TEAM\docling_eval
OUTPUT_DIR : D:\project\SKN29-FINAL-1TEAM\docling_eval\output
PDF_PATH   : D:\project\SKN29-FINAL-1TEAM\tiger_inc\pdf\법인카드_사용규정.pdf
size       : 612.5 KB

[pdf] 선택 가능한 PDF
  - tiger_inc/pdf/법인카드_사용규정.pdf
  - tiger_inc/pdf/부서소개.pdf
  - tiger_inc/pdf/업무추진비_사용규정.pdf
  - tiger_inc/pdf/조직도.pdf
  - tiger_inc/pdf/조직설계_상세기획서.pdf
  - tiger_inc/pdf/직급체계.pdf
  - tiger_inc/pdf/출장비_사용규정.pdf
  - tiger_inc/pdf/회식_운영규정.pdf

[law] 선택 가능한 PDF
  - tiger_inc/law/법인세법.pdf
  - tiger_inc/law/부가가치세법.pdf
  - tiger_inc/law/여신전문금융업법.pdf


---
## 3. Docling 문서 변환

파이프라인 옵션 두 가지가 이 평가의 전제다.

- `do_table_structure=True` + `do_cell_matching=True` — 표를 텍스트가 아니라 **셀 격자**로 복원한다(§6에서 검증).
- `heading_hierarchy_options.enabled=True` — 기본값 `False`에서는 모든 `section_header`가 `level=1`로
  **평탄화**되어 계층 검증(§5)이 무의미해진다. 활성화하면 제N장 → H1, 제N조 → H2 구조가 살아난다.

백엔드는 `pypdfium2` → 기본 `docling-parse` 순으로 시도한다(한글 PDF·한글 경로에서 기본 백엔드가
`'utf-8' codec can't decode ...`로 실패하는 사례가 있어 폴백을 둔다).

In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.do_ocr = False                       # 텍스트 PDF 기준. 스캔 PDF면 True
pipeline_options.heading_hierarchy_options.enabled = True   # ← 계층 검증의 핵심 스위치


def build_converter(backend=None):
    fmt = PdfFormatOption(pipeline_options=pipeline_options)
    if backend is not None:
        fmt.backend = backend
    return DocumentConverter(format_options={InputFormat.PDF: fmt})


result, used_backend = None, None
for _label, _backend in [("pypdfium2", PyPdfiumDocumentBackend), ("docling-parse (default)", None)]:
    try:
        _r = build_converter(_backend).convert(PDF_PATH)
        if _r.status in (ConversionStatus.SUCCESS, ConversionStatus.PARTIAL_SUCCESS):
            result, used_backend = _r, _label
            break
        print(f"[warn] backend={_label} -> status={_r.status}")
    except Exception as exc:  # noqa: BLE001
        print(f"[warn] backend={_label} -> {type(exc).__name__}: {exc}")

if result is None:
    raise RuntimeError("모든 PDF backend 조합이 실패했습니다. 위 [warn] 메시지를 확인하세요.")

doc = result.document

print("=" * 60)
print("Docling Document Conversion")
print("=" * 60)
print(f"PDF      : {PDF_PATH.name}")
print(f"Backend  : {used_backend}")
print(f"Pages    : {doc.num_pages()}")
print(f"Status   : {result.status.value.upper()}")
print(f"Errors   : {len(result.errors)}")
print(f"Objects  : texts={len(doc.texts)}  tables={len(doc.tables)}  "
      f"pictures={len(doc.pictures)}  groups={len(doc.groups)}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Docling Document Conversion
PDF      : 법인카드_사용규정.pdf
Backend  : pypdfium2
Pages    : 7
Status   : SUCCESS
Errors   : 0
Objects  : texts=118  tables=2  pictures=0  groups=16


---
## 3-1. 파싱 결함 후처리 (Post-processing)

변환 결과를 그대로 쓰면 안 되는 결함 6종이 이 규정 PDF들에서 재현된다.
원인은 docling과 원본 PDF 양쪽에 걸쳐 있고, 교정 로직은 `docling_eval/postprocess.py` 에 있다.

| 코드 | 증상 | 원인 | 복원 |
|---|---|---|---|
| R1 | 페이지 최상단 중앙정렬 헤딩(`제N장 …`)이 그 페이지 맨 뒤로 밀림 | docling 리딩오더 후처리 | 결정론적 |
| R2 | 목록 순번이 본문 끝으로 감 (`사적 용도의 지출 1.`) | 원본 PDF가 `<ol>` 마커를 콘텐츠 스트림 맨 뒤에 그림 | 결정론적 |
| R3 | 한 문단이 두 요소로 쪼개짐 (`… 관리자 승` / `인을 받아 …`) | R2의 마커가 문단 중간에 끼어들어 bbox 그룹핑 실패 | 결정론적 |
| R4 | 줄바꿈이 공백이 되어 어절이 깨짐 (`정산에 관 한 사항`) | CJK는 어절 중간에서 줄이 바뀌는데 docling은 공백으로 이어붙임 | **추정** |
| R5 | 자간 제목의 글자마다 공백 (`타 이 거 주 식 회 사`) | 원본이 글자 사이에 실제 공백 문자를 넣음 | 글리프 간격 |
| R6 | 두 항목이 한 덩어리로 뭉침 (`5. 여비교통비 … 6. 항목 구분이 …`) | R2의 마커가 문단 중간에 그려져 목록 경계를 못 잡음 | 결정론적 |

R4를 뺀 나머지는 결정론적으로 되돌아간다. **R4만 원리상 완전 복원이 불가능하다.** 양끝맞춤(justify) 조판이라 어절 내 줄바꿈과 어절 경계
줄바꿈의 글리프 간격이 구분되지 않고(측정값 3.4pt vs 3.5pt), 줄 끝 공백 글리프도 PDF에 남지 않아
정보 자체가 소실돼 있다. 그래서 **줄 경계에 닿지 않아 훼손되지 않은 토큰**으로 만든 어휘 사전에
괄호·연결부호·조사 규칙을 얹어 추정하고, 아래 셀에서 `tiger_inc/md/` 원본과 대조해 정확도를 함께 낸다.
(md 원본은 채점 전용이다 — RAG 런타임 경로가 아니다.)

**문서 유형에 따라 켤 단계가 다르다 (자동 판별).** R2~R3·R6는 "`<ol>` 마커가 콘텐츠 스트림 맨 뒤에
그려진다"는 규정 PDF 고유의 결함을 전제로 한다. 법령 PDF(국가법령정보센터)에는 그 결함이 아예 없어서
(페이지 끝 고아 마커 0건) **R3가 복구할 대상 없이 별개 조문·목을 붙여버린다** — 실측에서 법인세법 412건,
여신전문금융업법 126건을 오병합했고 `나. … 대금의 결제` + `다. 신용카드가맹점 …` → `결제다.` 처럼 뜻까지 바뀐다.
그래서 아래 셀은 **페이지 끝 고아 마커를 세어 규정형/법령형을 가르고**, 법령형이면 `steps={"R1","R4","R5"}`로
R2·R3·R6를 끈다. R1은 두 유형 모두 단(段)이 하나라 안전하고(본문 좌측 x가 한 값에 몰림), R4는 두 유형 모두 필요하다.

> R4는 이 문서군의 조판에 맞춰 튜닝돼 있다. 다른 조판·다른 언어 PDF에 적용할 때는
> 대조 점수를 먼저 확인할 것. R1도 **단(段)이 하나인 문서** 전제다.
> 도해·표 위주 문서(`조직도`·`조직설계_상세기획서`)는 R3가 트리 줄을 뭉개므로 후처리를 끄는 쪽이 낫다.


In [4]:
import importlib
import sys

import pypdfium2 as pdfium

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import postprocess

importlib.reload(postprocess)   # 모듈을 고쳐가며 다시 실행할 수 있게

ORPHAN_MARKER = re.compile(r"(?m)^\s*\d+[.)]\s*$")


def count_orphan_markers(pdf_path, probe_pages=12, tail_chars=160) -> int:
    """페이지 끝에 몰려 그려진 고아 목록 마커 수 — R2 결함의 지문."""
    raw = pdfium.PdfDocument(str(pdf_path))
    return sum(
        len(ORPHAN_MARKER.findall(raw[i].get_textpage().get_text_range()[-tail_chars:]))
        for i in range(min(len(raw), probe_pages))
    )


# 규정형(마커 있음) = R1~R6 전부 / 법령형(마커 0) = R2·R3·R6 끔 (§3-1 설명)
ORPHANS = count_orphan_markers(PDF_PATH)
DOC_KIND = "규정형" if ORPHANS else "법령형"
REPAIR_STEPS = None if ORPHANS else {"R1", "R4", "R5"}

# 어휘 사전은 같은 조판의 문서를 많이 볼수록 좋아진다 → 같은 폴더 PDF 전부 사용
# (규정+법령을 섞으면 규정 4종 합계가 3건 줄었다 — 폴더별로 따로 만드는 게 낫다)
VOCAB_PDFS = sorted(PDF_PATH.parent.glob("*.pdf")) or [PDF_PATH]
REFERENCE_MD = REPO_ROOT / "tiger_inc" / "md" / f"{PDF_PATH.stem}.md"   # 채점용(있으면)

print(f"[문서 유형] {DOC_KIND} — 페이지 끝 고아 마커 {ORPHANS}건 (앞 12p)")
print(f"[적용 단계] {'+'.join(s for s in postprocess.REPAIR_ORDER if REPAIR_STEPS is None or s in REPAIR_STEPS)}")
print()

before = {id(t): t.text for t in doc.texts}
report = postprocess.repair_document(doc, PDF_PATH, vocab_paths=VOCAB_PDFS, steps=REPAIR_STEPS)

print("[후처리 결과]")
for key, value in report.items():
    print(f"  {key:22}: {value}")

changed = [(before[id(t)], t.text) for t in doc.texts if before.get(id(t), t.text) != t.text]
print()
print(f"[텍스트가 바뀐 요소 {len(changed)}개 — 앞 5건]")
for old, new in changed[:5]:
    print(f"  - {old[:76]}")
    print(f"  + {new[:76]}")

print()
if REFERENCE_MD.exists():
    score = postprocess.score_line_joins(doc, PDF_PATH, REFERENCE_MD)
    total = score["exact"] + score["mismatch"]
    print(f"[md 원본 대조] {score['exact']}/{total} 요소 완전 일치")
    for sample in score["samples"]:
        print(f"  X {sample[:90]}")
    print("  ※ PDF가 md보다 최신 판이면(예: 제목 뒤 '개정 v1.1') 파싱 오류가 아니라 판 차이다.")
else:
    print(f"[md 원본 대조] 생략 — 정답지({REFERENCE_MD.name}) 없음")
    print("  → 자동 채점 대신 §3-2의 검수 시트로 판정 내역을 직접 확인한다.")


[문서 유형] 규정형 — 페이지 끝 고아 마커 26건 (앞 12p)
[적용 단계] R2+R3+R4+R5+R6+R1

[후처리 결과]
  steps                 : R2+R3+R4+R5+R6+R1
  R1_reordered          : 33
  R2_markers_restored   : 26
  R3_items_merged       : 4
  R4_texts_rejoined     : 24
  R4_unmatched          : 0
  R5_letter_spacing     : 1
  R6_items_split        : 1
  vocab_size            : 2235

[텍스트가 바뀐 요소 42개 — 앞 5건]
  - 타 이 거 주 식 회 사 ( T i ge r I n c . )
  + 타이거 주식회사 (Tiger Inc.)
  - 개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신 설, 별표
  + 개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신설, 별표1
  - ※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견
  + ※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견
  - 이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관 한 사항을 정함으로써 업무 
  + 이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관한 사항을 정함으로써 업무 효
  - "법인카드"란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한 다.
  + "법인카드"란 회사

### 3-2. 정답지 없는 문서 검수 시트 (법령)

`tiger_inc/md/`에 원본이 있는 규정 8종은 §3-1에서 자동 채점되지만, `tiger_inc/law/`의 법령 3종은
정답지가 없어 점수를 낼 수 없다. 대신 `docling_eval/review.py`가 **후처리가 내린 판정을 전수 기록**해
`output/review/<문서명>/`에 시트로 떨군다. 보는 순서는 다음과 같다.

| 파일 | 봐야 할 것 |
|---|---|
| `merges.csv` | R3가 별개 조문·목을 붙였는지 (법령형은 R3를 껐으므로 비어 있어야 정상) |
| `linebreaks_weak.csv` | 근거 없이 길이로 찍은 R4 줄바꿈 판정 — `decision`이 맞는지 |
| `unmatched.csv` | 원본 대조에 실패해 R4가 손대지 못한 요소 |
| `result.md` | RAG에 실제로 들어갈 최종 텍스트 |

`summary.md`의 **weak 비율**이 판단 기준이다 — 실측은 법인세법 28%, 부가가치세법 27%,
여신전문금융업법 44%로, 여신법은 줄 경계의 절반 가까이가 추측이라 R4를 신뢰하기 어렵다.

시트 생성은 PDF를 한 번 더 변환하므로(97p 법인세법 기준 수 분) 기본은 꺼 두었다. 터미널에서 돌리는 편이 빠르다.

```bash
python review.py                            # law/ 3종 전부
python review.py 법인세법 --steps R1,R4,R5    # R3 끄고 하나만
python review.py 법인세법 --out output/review_all   # 비교용 다른 폴더로
```


In [5]:
RUN_REVIEW = False          # True → 이 자리에서 시트 생성 (PDF 재변환 — 수 분)
REVIEW_ROOT = OUTPUT_DIR / "review"
REVIEW_DIR = REVIEW_ROOT / PDF_PATH.stem

if RUN_REVIEW:
    import review

    importlib.reload(review)
    summary = review.build_review(
        PDF_PATH, REVIEW_ROOT, vocab_paths=VOCAB_PDFS, steps=REPAIR_STEPS
    )
    for key, value in summary.items():
        print(f"  {key:22}: {value}")
    print()

if (REVIEW_DIR / "summary.md").exists():
    print(f"[검수 시트] {REVIEW_DIR}")
    for p in sorted(REVIEW_DIR.iterdir()):
        print(f"  {p.name:22} {p.stat().st_size:>9,} bytes")
    display(Markdown((REVIEW_DIR / "summary.md").read_text(encoding="utf-8")))
else:
    _steps = "" if REPAIR_STEPS is None else f" --steps {','.join(sorted(REPAIR_STEPS))}"
    print(f"검수 시트 없음 — RUN_REVIEW=True 로 두고 다시 실행하거나, 터미널에서:")
    print(f"  python review.py {PDF_PATH.stem}{_steps}")


검수 시트 없음 — RUN_REVIEW=True 로 두고 다시 실행하거나, 터미널에서:
  python review.py 법인카드_사용규정


---
## 4. Layout Analysis 검증

> **Docling이 PDF 페이지의 서로 다른 영역을 올바른 문서 요소로 구분하는가?**

`doc.iterate_items()`로 문서 트리를 읽기 순서대로 순회하며 각 요소의
**페이지 · 라벨 · 계층 깊이 · Bounding Box · 텍스트 · 순서**를 뽑는다.
Docling의 세부 라벨(`section_header`, `list_item`, `page_header` …)은 비교하기 쉽도록
`Title / Heading / Text / List / Table / Picture / Caption / Header / Footer` 등 큰 범주로 묶는다.

In [6]:
LABEL_GROUP = {
    DocItemLabel.TITLE: "Title",
    DocItemLabel.SECTION_HEADER: "Heading",
    DocItemLabel.TEXT: "Text",
    DocItemLabel.PARAGRAPH: "Text",
    DocItemLabel.LIST_ITEM: "List",
    DocItemLabel.TABLE: "Table",
    DocItemLabel.PICTURE: "Picture",
    DocItemLabel.CHART: "Picture",
    DocItemLabel.CAPTION: "Caption",
    DocItemLabel.PAGE_HEADER: "Header",
    DocItemLabel.PAGE_FOOTER: "Footer",
    DocItemLabel.FOOTNOTE: "Footnote",
    DocItemLabel.FORMULA: "Formula",
    DocItemLabel.CODE: "Code",
    DocItemLabel.DOCUMENT_INDEX: "Index",
}


def group_of(label):
    """Docling 세부 라벨 -> 평가용 큰 범주."""
    if label in LABEL_GROUP:
        return LABEL_GROUP[label]
    return str(getattr(label, "value", label)).replace("_", " ").title()


def text_of(item):
    """번호 목록의 순번은 text 가 아니라 ListItem.marker 에 따로 저장되므로 다시 합쳐서 보여준다."""
    text = (getattr(item, "text", "") or "").strip()
    marker = (getattr(item, "marker", "") or "").strip()
    return f"{marker} {text}" if marker else text


def prov_of(item):
    """첫 번째 provenance(페이지 + bbox). 없으면 None."""
    prov = getattr(item, "prov", None) or []
    return prov[0] if prov else None


layout_rows = []
for order, (item, depth) in enumerate(doc.iterate_items(with_groups=False), start=1):
    label = getattr(item, "label", None)
    if label is None:          # 그룹 노드 등 라벨 없는 항목은 건너뜀
        continue
    prov = prov_of(item)
    bbox = prov.bbox if prov is not None else None
    text = (
        f"<table {item.data.num_rows}x{item.data.num_cols}>"
        if isinstance(item, TableItem)
        else text_of(item)
    )
    layout_rows.append(
        {
            "Order": order,
            "Page": prov.page_no if prov is not None else None,
            "Label": str(getattr(label, "value", label)),
            "Element Type": group_of(label),
            "Depth": depth,
            "Level": getattr(item, "level", None),
            "BBox(l,t,r,b)": (
                f"({bbox.l:.0f},{bbox.t:.0f},{bbox.r:.0f},{bbox.b:.0f})" if bbox is not None else ""
            ),
            "Text": text,
        }
    )

layout_df = pd.DataFrame(layout_rows)
print(f"추출된 문서 요소: {len(layout_df)} 개")
display(layout_df.head(20))

추출된 문서 요소: 99 개


,Order,Page,Label,Element Type,Depth,Level,"BBox(l,t,r,b)",Text
0,1,1,section_header,Heading,1,1.0,"(206,594,388,580)",타이거 주식회사 (Tiger Inc.)
1,2,1,section_header,Heading,1,1.0,"(183,512,409,486)",법인카드 사용 규정
2,3,1,table,Table,1,NaN,"(205,379,390,293)",<table 3x2>
3,4,1,text,Text,1,NaN,"(113,272,481,251)","개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신설, 별표1..."
4,5,1,text,Text,1,NaN,"(91,96,504,63)","※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견..."
5,6,2,section_header,Heading,1,1.0,"(265,717,329,703)",제1장 총칙
6,7,2,section_header,Heading,1,2.0,"(63,667,121,655)",제1조 (목적)
7,8,2,text,Text,1,NaN,"(63,641,533,597)","이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관한 사항을 정함으로써 업무 효..."
8,9,2,section_header,Heading,1,2.0,"(63,572,160,560)",제2조 (정의) 개정 v1.1
9,10,2,text,Text,1,NaN,"(63,546,279,536)",이 규정에서 사용하는 용어의 정의는 다음과 같다.


### 4-1. 페이지별 / 전체 요소 집계

In [7]:
# 페이지 × 요소 타입
page_counts_df = (
    layout_df.groupby(["Page", "Element Type"])
    .size()
    .reset_index(name="Count")
    .sort_values(["Page", "Element Type"])
    .reset_index(drop=True)
)
print("[페이지별 요소 개수]")
display(page_counts_df)

# 한눈에 보는 교차표
layout_pivot_df = (
    layout_df.pivot_table(index="Page", columns="Element Type", values="Order", aggfunc="count")
    .fillna(0)
    .astype(int)
)
print("[페이지 × 요소 타입 교차표]")
display(layout_pivot_df)

# 문서 전체 집계
element_totals_df = (
    layout_df["Element Type"].value_counts().rename_axis("Element Type").reset_index(name="Count")
)
print("[문서 전체 집계]")
display(element_totals_df)

layout_csv = LAYOUT_DIR / "layout_result.csv"
layout_df.to_csv(layout_csv, index=False, encoding="utf-8-sig")  # Excel 한글 대비 utf-8-sig
print(f"saved -> {layout_csv}")

[페이지별 요소 개수]


,Page,Element Type,Count
0,1,Heading,2
1,1,Table,1
2,1,Text,2
3,2,Heading,4
4,2,List,6
5,2,Text,3
6,3,Heading,7
7,3,List,10
8,3,Text,3
9,4,Heading,4


[페이지 × 요소 타입 교차표]


Element Type,Caption,Heading,List,Table,Text
Page,,,,,
1,0,2,0,1,2
2,0,4,6,0,3
3,0,7,10,0,3
4,0,4,16,0,0
5,0,5,12,0,1
6,0,6,9,0,1
7,1,0,5,1,0


[문서 전체 집계]


,Element Type,Count
0,List,58
1,Heading,28
2,Text,10
3,Table,2
4,Caption,1


saved -> D:\project\SKN29-FINAL-1TEAM\docling_eval\output\layout\layout_result.csv


---
## 5. Text Hierarchy 검증

> **텍스트를 뽑는 데 그치지 않고 문서의 논리적 계층 구조를 유지하는가?**

확인 항목: 제목/본문 구분 · Heading 레벨 · Heading→Paragraph 관계 · Section/Subsection 구조 ·
요소 순서 · 목록 구조 · 번호 체계.

Heading 레벨은 `SectionHeaderItem.level`(§3에서 `heading_hierarchy_options`를 켰기 때문에 실제 값이 들어온다),
`title` 라벨은 H1로 취급한다.

In [8]:
HEADING_LABELS = {DocItemLabel.TITLE, DocItemLabel.SECTION_HEADER}
BODY_TYPES = {"Text", "List", "Table", "Picture", "Caption", "Formula", "Code"}


def heading_level(item):
    if getattr(item, "label", None) == DocItemLabel.TITLE:
        return 1
    return int(getattr(item, "level", 1) or 1)


hier_rows = []
for order, (item, _depth) in enumerate(doc.iterate_items(with_groups=False), start=1):
    label = getattr(item, "label", None)
    if label is None:
        continue
    prov = prov_of(item)
    if label in HEADING_LABELS:
        row_type, level, text = "Heading", f"H{heading_level(item)}", text_of(item)
    elif isinstance(item, TableItem):
        row_type, level = "Table", "-"
        text = f"<table {item.data.num_rows}x{item.data.num_cols}>"
    elif label == DocItemLabel.LIST_ITEM:
        row_type, level, text = "List", "-", text_of(item)
    elif label == DocItemLabel.PICTURE:
        row_type, level, text = "Picture", "-", "<picture>"
    else:
        row_type, level, text = group_of(label), "-", text_of(item)
    hier_rows.append(
        {
            "Order": order,
            "Page": prov.page_no if prov is not None else None,
            "Type": row_type,
            "Level": level,
            "Text": text,
        }
    )

hierarchy_df = pd.DataFrame(hier_rows)
display(hierarchy_df.head(40))

,Order,Page,Type,Level,Text
0,1,1,Heading,H1,타이거 주식회사 (Tiger Inc.)
1,2,1,Heading,H1,법인카드 사용 규정
2,3,1,Table,-,<table 3x2>
3,4,1,Text,-,"개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신설, 별표1..."
4,5,1,Text,-,"※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견..."
5,6,2,Heading,H1,제1장 총칙
6,7,2,Heading,H2,제1조 (목적)
7,8,2,Text,-,"이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관한 사항을 정함으로써 업무 효..."
8,9,2,Heading,H2,제2조 (정의) 개정 v1.1
9,10,2,Text,-,이 규정에서 사용하는 용어의 정의는 다음과 같다.


### 5-1. 문서 계층 트리

In [9]:
def shorten(text, width=60):
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 1] + "…"


def render_hierarchy(df, max_body_per_heading=3, max_lines=150):
    """Heading 레벨 기준 들여쓰기 + 하위 본문 미리보기로 트리를 그린다."""
    lines, pending, indent = [], [], ""

    def flush():
        """모아둔 본문을 ├─ / └─ 로 출력."""
        shown = pending[:max_body_per_heading]
        rest = len(pending) - len(shown)
        for i, (typ, text) in enumerate(shown):
            last = (i == len(shown) - 1) and rest == 0
            lines.append(f"{indent}{'└─' if last else '├─'} [{typ.upper()}] {shorten(text)}")
        if rest:
            lines.append(f"{indent}└─ … ({rest} more)")
        pending.clear()

    for _, row in df.iterrows():
        if row["Type"] == "Heading":
            flush()
            level = int(row["Level"][1:])
            lines.append(f"{'    ' * (level - 1)}[{row['Level']}] {shorten(row['Text'], 70)}")
            indent = "    " * level
        elif row["Type"] in BODY_TYPES:
            pending.append((row["Type"], row["Text"]))
    flush()

    print("DOCUMENT HIERARCHY")
    print("=" * 64)
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} lines omitted)")


render_hierarchy(hierarchy_df)

DOCUMENT HIERARCHY
[H1] 타이거 주식회사 (Tiger Inc.)
[H1] 법인카드 사용 규정
    ├─ [TABLE] <table 3x2>
    ├─ [TEXT] 개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 …
    └─ [TEXT] ※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으…
[H1] 제1장 총칙
    [H2] 제1조 (목적)
        └─ [TEXT] 이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 …
    [H2] 제2조 (정의) 개정 v1.1
        ├─ [TEXT] 이 규정에서 사용하는 용어의 정의는 다음과 같다.
        ├─ [LIST] 1. "법인카드"란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한다.
        ├─ [LIST] 2. "사용자"란 법인카드를 발급받아 사용하는 임직원을 말한다.
        └─ … (4 more)
    [H2] 제3조 (적용범위)
        └─ [TEXT] 이 규정은 회사로부터 법인카드를 발급받은 모든 임직원에게 적용한다.
[H1] 제2장 법인카드의 발급 및 관리
    [H2] 제4조 (발급 대상)
        ├─ [LIST] 1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발급한…
        └─ [LIST] 2. 직책이 없는 임직원(비직책자)은 부서 공용카드를 사용하거나, 업무상 필요가 인정되는 경우 관리자 승인…
    [H2] 제5조 (발급 절차)
        └─ [TEXT] 사용자는 소속 부서장의 승인을 받아 「법인카드 발급 신청서」를 경영지원본부에 제출하며, 경영지원본부는 신청…
    [H2] 제6조 (관리 책임)
        ├─ [LIST] 1. 사용자는 법

### 5-2. 계층 통계 (Heading 레벨 분포·건전성)

In [10]:
heading_df = hierarchy_df[hierarchy_df["Type"] == "Heading"].copy()
heading_levels = [int(v[1:]) for v in heading_df["Level"]]

heading_level_df = (
    heading_df["Level"].value_counts().rename_axis("Level").reset_index(name="Count").sort_values("Level")
)
display(heading_level_df)

# 레벨이 2단계 이상 건너뛰면(H1 -> H3) 계층 인식이 깨진 신호
level_skips = sum(1 for a, b in zip(heading_levels, heading_levels[1:]) if b - a > 1)
body_count = int(hierarchy_df["Type"].isin(BODY_TYPES).sum())

print(f"Headings detected  : {len(heading_levels)}")
print(f"Max heading depth  : {max(heading_levels) if heading_levels else 0}")
print(f"Level skips (>1)   : {level_skips}   (0 이면 계층이 자연스럽게 이어짐)")
print(f"Body elements      : {body_count}")
display(heading_df.head(30))

,Level,Count
1,H1,9
0,H2,19


Headings detected  : 28
Max heading depth  : 2
Level skips (>1)   : 0   (0 이면 계층이 자연스럽게 이어짐)
Body elements      : 71


,Order,Page,Type,Level,Text
0,1,1,Heading,H1,타이거 주식회사 (Tiger Inc.)
1,2,1,Heading,H1,법인카드 사용 규정
5,6,2,Heading,H1,제1장 총칙
6,7,2,Heading,H2,제1조 (목적)
8,9,2,Heading,H2,제2조 (정의) 개정 v1.1
16,17,2,Heading,H2,제3조 (적용범위)
18,19,3,Heading,H1,제2장 법인카드의 발급 및 관리
19,20,3,Heading,H2,제4조 (발급 대상)
22,23,3,Heading,H2,제5조 (발급 절차)
24,25,3,Heading,H2,제6조 (관리 책임)


### 5-3. 리스트 마커 복원 검증

번호 목록(`1.`, `2.` …)의 순번은 본문 텍스트가 아니라 `ListItem.marker` 필드에 분리 저장된다.
그런데 이 PDF들은 HTML→PDF 변환기가 `<ol>` 마커를 **콘텐츠 스트림 맨 뒤에** 따로 그려 두어
(페이지 끝에 `1. 2. 3. …`이 몰려 있다) docling이 마커를 인식하지 못하고
순번이 본문 끝에 `… 발급한다. 1.` 처럼 밀려 붙는다.

§3-1의 **R2**가 이것을 되돌린다. 아래 지표에서 `순번이 본문 끝으로`가 **0**이고
`marker 정상 인식`이 전체와 같으면 복원이 끝난 것이다.

In [11]:
TRAILING_NUM = re.compile(r"\s\d+[.)]\s*$")   # 본문 끝으로 밀려난 순번

list_items = [it for it, _ in doc.iterate_items(with_groups=False) if isinstance(it, ListItem)]
marker_rows = []
for it in list_items:
    prov = prov_of(it)
    raw_text = (it.text or "")
    marker_rows.append(
        {
            "Page": prov.page_no if prov is not None else None,
            "Marker": (it.marker or "").strip(),
            "Marker OK": bool((it.marker or "").strip()),
            "Trailing num": bool(TRAILING_NUM.search(raw_text)),
            "Text": raw_text[:70],
        }
    )

marker_df = pd.DataFrame(
    marker_rows, columns=["Page", "Marker", "Marker OK", "Trailing num", "Text"]
)
if marker_df.empty:
    print("번호 목록(list_item)이 없습니다.")
else:
    n_ok = int(marker_df["Marker OK"].sum())
    n_tail = int(marker_df["Trailing num"].sum())
    print(f"list_item          : {len(marker_df)}")
    print(f"marker 정상 인식   : {n_ok}")
    print(f"marker 인식 실패   : {len(marker_df) - n_ok}")
    if REPAIR_STEPS is None or "R2" in REPAIR_STEPS:
        print(f"순번이 본문 끝으로 : {n_tail}   ← §3-1 R2 적용 후 0이어야 정상")
    else:
        print(f"순번이 본문 끝으로 : {n_tail}   ← R2 미적용({DOC_KIND}) — 원래부터 0이어야 정상")
    display(
        marker_df.groupby("Page")[["Marker OK", "Trailing num"]].sum().astype(int)
    )
    broken = marker_df[~marker_df["Marker OK"]]
    if not broken.empty:
        print("\n[마커 인식 실패 예시]")
        display(broken.head(10))

list_item          : 58
marker 정상 인식   : 51
marker 인식 실패   : 7
순번이 본문 끝으로 : 0   ← §3-1 R2 적용 후 0이어야 정상


,Marker OK,Trailing num
Page,,
2,6,0
3,10,0
4,16,0
5,12,0
6,7,0
7,0,0



[마커 인식 실패 예시]


,Page,Marker,Marker OK,Trailing num,Text
51,6,,False,False,이 규정의 개정은 경영지원본부의 제안으로 대표이사의 승인을 받아 시행한다.
52,6,,False,False,이 규정은 2026년 8월 1일부터 시행한다.
53,7,,False,False,"※ 한도는 보임 중인 직책 기준으로 적용되며, 직급(사원~전무)과는 무관하다. 직책이 변경된 경우 한도는 발령일부터 적용된다."
54,7,,False,False,"※ 경영지원본부는 본부장 직위가 없으므로, 산하 각 부서장(인사부·재무회계부·총무구매부·법무부·IT운영부)은 부서장 한도를 적"
55,7,,False,False,"※ 식대·기업업무추진비 지출은 제10조 제2항에 따라 직책과 무관하게 건당 30만원 초과 시 사전승인 대상이며, 위 표의 직책"
56,7,,False,False,"※ ""이사""가 부서장을 겸직하는 경우 부서장 한도를, 본부장 대행으로 보임된 경우 본부장 한도를 적용한다. 겸직·대행 여부 및"
57,7,,False,False,"※ ""전무""가 복수본부 총괄로 보임된 경우에도 본부장 한도를 적용하며, 발령 문서상 총괄 대상 본부가 둘 이상인 경우 카드는"


---
## 6. Table Structure 검증

> **원본 PDF의 표 구조가 구조화된 데이터로 얼마나 정확하게 복원되는가?**

표마다 `TableData.table_cells`를 직접 들여다보고 **행/열 수 · 셀 수 · 헤더(열/행) 감지 ·
병합 셀(`row_span`/`col_span` > 1) · 빈 셀**을 집계한다.
그다음 Markdown 렌더링과 `pandas.DataFrame` 변환 결과를 나란히 확인하고,
셀 단위 원본 구조는 `table_NN.json`, 표 형태는 `table_NN.csv`로 저장한다.

In [12]:
table_summaries = []
table_frames = {}

if not doc.tables:
    print("이 PDF에서 표가 감지되지 않았습니다. (Tables detected: 0)")

for idx, table in enumerate(doc.tables, start=1):
    data = table.data
    prov = prov_of(table)
    cells = data.table_cells
    merged = [c for c in cells if c.row_span > 1 or c.col_span > 1]
    empties = [c for c in cells if not (c.text or "").strip()]
    col_headers = [c for c in cells if c.column_header]
    row_headers = [c for c in cells if c.row_header]
    header_rows = sorted({c.start_row_offset_idx for c in col_headers})

    table_df = table.export_to_dataframe(doc)
    table_frames[idx] = table_df

    print("=" * 64)
    print(f"TABLE {idx}")
    print("=" * 64)
    print(f"Page        : {prov.page_no if prov is not None else '-'}")
    print(f"Rows        : {data.num_rows}")
    print(f"Columns     : {data.num_cols}")
    print(f"Cells       : {len(cells)}")
    print(f"Header      : {f'Detected (row {header_rows})' if col_headers else 'Not detected'}")
    print(f"Row header  : {f'Detected ({len(row_headers)} cells)' if row_headers else 'Not detected'}")
    print(f"Merged cell : {f'Detected ({len(merged)})' if merged else 'None'}")
    print(f"Empty cell  : {len(empties)}")
    print(f"Caption     : {table.caption_text(doc) or '-'}")
    print(f"DataFrame   : {table_df.shape[0]} rows x {table_df.shape[1]} cols")

    print("\n[Markdown 렌더링]")
    display(Markdown(table.export_to_markdown(doc)))
    print("[DataFrame]")
    display(table_df)

    payload = {
        "index": idx,
        "page": prov.page_no if prov is not None else None,
        "caption": table.caption_text(doc),
        "num_rows": data.num_rows,
        "num_cols": data.num_cols,
        "num_cells": len(cells),
        "header_rows": header_rows,
        "num_column_header_cells": len(col_headers),
        "num_row_header_cells": len(row_headers),
        "num_merged_cells": len(merged),
        "num_empty_cells": len(empties),
        "cells": [
            {
                "text": c.text,
                "row": [c.start_row_offset_idx, c.end_row_offset_idx],
                "col": [c.start_col_offset_idx, c.end_col_offset_idx],
                "row_span": c.row_span,
                "col_span": c.col_span,
                "column_header": c.column_header,
                "row_header": c.row_header,
            }
            for c in cells
        ],
    }
    (TABLE_DIR / f"table_{idx:02d}.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    table_df.to_csv(TABLE_DIR / f"table_{idx:02d}.csv", index=False, encoding="utf-8-sig")

    table_summaries.append(
        {
            "Table": idx,
            "Page": payload["page"],
            "Rows": data.num_rows,
            "Cols": data.num_cols,
            "Cells": len(cells),
            "Header": bool(col_headers),
            "Merged": len(merged),
            "Empty": len(empties),
            "DF Shape": f"{table_df.shape[0]}x{table_df.shape[1]}",
        }
    )

print(f"\nsaved -> {TABLE_DIR}")

TABLE 1
Page        : 1
Rows        : 3
Columns     : 2
Cells       : 6
Header      : Not detected
Row header  : Not detected
Merged cell : None
Empty cell  : 0
Caption     : -
DataFrame   : 3 rows x 2 cols

[Markdown 렌더링]


| 제정일    | 2026. 7. 20.   |
|-----------|----------------|
| 시행일    | 2026. 8. 1.    |
| 소관부서  | 경영지원본부   |

[DataFrame]


,0,1
0,제정일,2026. 7. 20.
1,시행일,2026. 8. 1.
2,소관부서,경영지원본부


TABLE 2
Page        : 7
Rows        : 6
Columns     : 4
Cells       : 24
Header      : Detected (row [0])
Row header  : Not detected
Merged cell : None
Empty cell  : 0
Caption     : 별표 1. 직책별 법인카드 사용 한도 개정 v1.1
DataFrame   : 5 rows x 4 cols

[Markdown 렌더링]


별표 1. 직책별 법인카드 사용 한도 개정 v1.1

| 직책               | 1일 한도    | 월 한도    | 건당 사전승인 기준   |
|--------------------|-------------|------------|----------------------|
| 대표이사           | 300만원     | 1,000만원  | 100만원 초과         |
| 본부장             | 250만원     | 800만원    | 80만원 초과          |
| 부서장             | 150만원     | 500만원    | 60만원 초과          |
| 팀장               | 100만원     | 400만원    | 50만원 초과          |
| 비직책자(공용카드) | 50만원      | 200만원    | 30만원 초과          |

[DataFrame]


,직책,1일 한도,월 한도,건당 사전승인 기준
0,대표이사,300만원,"1,000만원",100만원 초과
1,본부장,250만원,800만원,80만원 초과
2,부서장,150만원,500만원,60만원 초과
3,팀장,100만원,400만원,50만원 초과
4,비직책자(공용카드),50만원,200만원,30만원 초과



saved -> D:\project\SKN29-FINAL-1TEAM\docling_eval\output\tables


### 6-1. 표 구조 복원 요약

In [13]:
table_summary_df = pd.DataFrame(
    table_summaries,
    columns=["Table", "Page", "Rows", "Cols", "Cells", "Header", "Merged", "Empty", "DF Shape"],
)
if table_summary_df.empty:
    print("표가 없어 요약할 내용이 없습니다.")
else:
    display(table_summary_df)
    print(f"Tables detected  : {len(doc.tables)}")
    print(f"Tables converted : {len(table_frames)}")
    print(f"Rows total       : {int(table_summary_df['Rows'].sum())}")
    print(f"Cells total      : {int(table_summary_df['Cells'].sum())}")
    print(f"With header      : {int(table_summary_df['Header'].sum())} / {len(table_summary_df)}")
    print(f"Merged cells     : {int(table_summary_df['Merged'].sum())}")

,Table,Page,Rows,Cols,Cells,Header,Merged,Empty,DF Shape
0,1,1,3,2,6,False,0,0,3x2
1,2,7,6,4,24,True,0,0,5x4


Tables detected  : 2
Tables converted : 2
Rows total       : 9
Cells total      : 30
With header      : 1 / 2
Merged cells     : 0


---
## 7. 전체 결과 요약

세 평가 영역(Layout / Hierarchy / Table)을 한 화면에서 확인한다.

In [14]:
counts = layout_df["Element Type"].value_counts().to_dict()
tables_total_rows = int(table_summary_df["Rows"].sum()) if not table_summary_df.empty else 0
tables_total_cols = int(table_summary_df["Cols"].sum()) if not table_summary_df.empty else 0
tables_merged = int(table_summary_df["Merged"].sum()) if not table_summary_df.empty else 0

print("=" * 64)
print("DOCLING PARSING TEST SUMMARY")
print("=" * 64)
print(f"PDF                : {PDF_PATH.name}")
print(f"Backend            : {used_backend}")
print(f"Status             : {result.status.value.upper()}")

print("\n[1] Layout Analysis")
print("-" * 32)
print(f"{'Pages':19}: {doc.num_pages()}")
print(f"{'Elements':19}: {len(layout_df)}")
for key in ["Title", "Heading", "Text", "List", "Table", "Picture", "Caption", "Header", "Footer"]:
    print(f"{key:19}: {int(counts.get(key, 0))}")

print("\n[2] Text Hierarchy")
print("-" * 32)
print(f"{'Heading detected':19}: {len(heading_levels)}")
for lvl in sorted(set(heading_levels)):
    print(f"{'H' + str(lvl):19}: {heading_levels.count(lvl)}")
print(f"{'Max depth':19}: {max(heading_levels) if heading_levels else 0}")
print(f"{'Level skips':19}: {level_skips}")
print(f"{'Body elements':19}: {body_count}")

print("\n[3] Table Structure")
print("-" * 32)
print(f"{'Tables detected':19}: {len(doc.tables)}")
print(f"{'Tables converted':19}: {len(table_frames)}")
print(f"{'Rows reconstructed':19}: {tables_total_rows}")
print(f"{'Columns detected':19}: {tables_total_cols}")
print(f"{'Merged cells':19}: {tables_merged}")

summary_df = pd.DataFrame(
    [
        ("1. Layout Analysis", "Pages", doc.num_pages()),
        ("1. Layout Analysis", "Elements", len(layout_df)),
        ("1. Layout Analysis", "Distinct element types", layout_df["Element Type"].nunique()),
        ("1. Layout Analysis", "Tables / Pictures", f"{int(counts.get('Table', 0))} / {int(counts.get('Picture', 0))}"),
        ("2. Text Hierarchy", "Headings", len(heading_levels)),
        ("2. Text Hierarchy", "Max heading depth", max(heading_levels) if heading_levels else 0),
        ("2. Text Hierarchy", "Level skips (>1)", level_skips),
        ("2. Text Hierarchy", "Body elements", body_count),
        ("3. Table Structure", "Tables detected", len(doc.tables)),
        ("3. Table Structure", "Tables converted", len(table_frames)),
        ("3. Table Structure", "Rows reconstructed", tables_total_rows),
        ("3. Table Structure", "Merged cells", tables_merged),
    ],
    columns=["Area", "Metric", "Value"],
)
display(summary_df)

DOCLING PARSING TEST SUMMARY
PDF                : 법인카드_사용규정.pdf
Backend            : pypdfium2
Status             : SUCCESS

[1] Layout Analysis
--------------------------------
Pages              : 7
Elements           : 99
Title              : 0
Heading            : 28
Text               : 10
List               : 58
Table              : 2
Picture            : 0
Caption            : 1
Header             : 0
Footer             : 0

[2] Text Hierarchy
--------------------------------
Heading detected   : 28
H1                 : 9
H2                 : 19
Max depth          : 2
Level skips        : 0
Body elements      : 71

[3] Table Structure
--------------------------------
Tables detected    : 2
Tables converted   : 2
Rows reconstructed : 9
Columns detected   : 6
Merged cells       : 0


,Area,Metric,Value
0,1. Layout Analysis,Pages,7
1,1. Layout Analysis,Elements,99
2,1. Layout Analysis,Distinct element types,5
3,1. Layout Analysis,Tables / Pictures,2 / 0
4,2. Text Hierarchy,Headings,28
5,2. Text Hierarchy,Max heading depth,2
6,2. Text Hierarchy,Level skips (>1),0
7,2. Text Hierarchy,Body elements,71
8,3. Table Structure,Tables detected,2
9,3. Table Structure,Tables converted,2


---
## 8. Markdown 결과 저장

Docling이 복원한 문서를 Markdown으로 export하여 `docling_eval/output/docling_result.md`에 저장하고,
앞부분을 노트북에서 바로 확인한다. (§4~6의 구조 판단을 눈으로 교차검증하는 용도)

In [15]:
markdown_text = doc.export_to_markdown()
md_path = OUTPUT_DIR / "docling_result.md"
md_path.write_text(markdown_text, encoding="utf-8")
print(f"saved -> {md_path}  ({len(markdown_text):,} chars)")

preview = markdown_text[:2000]
display(Markdown(preview + ("\n\n… (이하 생략)" if len(markdown_text) > 2000 else "")))

saved -> D:\project\SKN29-FINAL-1TEAM\docling_eval\output\docling_result.md  (6,181 chars)


## 타이거 주식회사 (Tiger Inc.)

## 법인카드 사용 규정

| 제정일    | 2026. 7. 20.   |
|-----------|----------------|
| 시행일    | 2026. 8. 1.    |
| 소관부서  | 경영지원본부   |

개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사전승인 예외절차 신설, 별표1 이사 겸직·대행 시 한도 적용기준 명확화

※ 본 규정은 데모/교육 목적의 가상 기업(타이거 주식회사, Tiger Inc.) 시나리오용으로 작성되었으며, 회사는 설립 12년차 중견기업(세법상 중소기업 특례 대상 아님)임을 가정합니다. 연 매출액은 연결 기준 약 1,200억원(500억원 초과 구간)으로 가정하였으며, 실제 적용 시 회사의 정확한 매출 규모·업종·지주회사 여부 등에 따라 한도 및 조항을 조정해야 합니다.

## 제1장 총칙

### 제1조 (목적)

이 규정은 타이거 주식회사(이하 "회사")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관한 사항을 정함으로써 업무 효율성을 높이고, 관계 법령에 따른 손금 인정 요건을 충족하여 세무상 불이익을 방지하며, 법인카드 오남용을 예방하는 것을 목적으로 한다.

### 제2조 (정의) 개정 v1.1

이 규정에서 사용하는 용어의 정의는 다음과 같다.

1. "법인카드"란 회사 명의로 발급되어 임직원이 업무 목적의 지출에 사용하는 신용카드·체크카드를 말한다.
2. "사용자"란 법인카드를 발급받아 사용하는 임직원을 말한다.
3. "관리자"란 법인카드 정산 승인 권한을 가진 자로서 「직급체계」에 따라 결재권을 보유한 팀장·부서장·본부장 및 대표이사를 말한다. 법인카드의 발급·회수 등 관리업무는 경영지원본부(인사부·재무회계부·총무구매부·법무부)가 총괄한다.
4. "담당자"란 법인카드를 발급받아 사용하고 정산을 신청하는 임직원 본인을 말하며, 법인카드 사용 신청·영수증 업로드·사유 작성 및 반려 시 재상신·이의제기(어필)를 수행하는 자를 말한다(제2호의 "사용자"와 동일인을 지칭하는 정산 시스템상의 역할 명칭이다). 정산 서류의 1차 확인·처리 업무는 재무회계부(재무회계팀)가 수행한다.
5. "기업업무추진비"란 회사가 업무와 관련하여 접대·교제·사례 등의 목적으로 지출하는 비용을 말한다(종전 "접대비").
6. "적격증명서류"란 신용카드매출전표, 현금영수증, 세금계산서, 계산서 등 세법상 인정되는 증빙서류를 말한다.

### 제3조 (적용범위)

이 규정은 회사로부터 법인카드를 발급받은 모든 임직원에게 적용한다.

## 제2장 법인카드의 발급 및 관리

### 제4조 (발급 대상)

1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발급한다.
2. 직책이 없는 임직원(비직책자)은 부서 공용카드를 사용하거나, 업무상 필요가 인정되는 경우 관리자 승인을 받아 개인 카드를 발급받을 수 있다.

### 제5조 (발급 절차)

사용자는 소속 부서장의 승인을 받아 「법인카드 발급 신청서」를 경영지원본부에 제출하며, 경영지원본부는 신청 내용을 검토하여 카드사에 발급을 요청한다.

### 제6조 (관리 책임)

1. 사용자는 법인카드를 선량한 관리자의 주의로 보관·사용하여야 하며, 타인에게 양도·대여할 수 없다. 다만 제4조제2항의 부서 공용카드는 소속 부서장의 관리 하에 부서원이 순환하여 사용할 수 있으며, 이는 제1항의 양도·대여 금지 대상에 해당하지 아니한다. 이 경우 정산 시스템에 실사용자를 반드시 기록하여야 한다.
2. 퇴사, 휴직, 부서 이동 시 사용자는 지체 없이 법인카드를 반납하여야 한다.

### 제7조 (분실·도난 시 조치)

사용자는 법인카드의 분실 또는 도난을 인지한 즉시 카드사에 사용정지를 요청하고, 24시간 이내에 경영지원본부에 서면(이메일 포함)으로 보고하여야 한다.

## 제3장 법인카드 사용 원칙

### 제8조 (사용 가능 항목)

법인카드는 다음 각 호의 업무 목적 지출에 한하여 사용할 

… (이하 생략)

In [16]:
# 생성된 산출물 목록
print(f"{OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUTPUT_DIR).as_posix():30} {p.stat().st_size:>9,} bytes")

D:\project\SKN29-FINAL-1TEAM\docling_eval\output
  docling_result.md                 13,749 bytes
  layout/layout_result.csv          17,387 bytes
  review/법인세법/elements.csv         442,764 bytes
  review/법인세법/linebreaks.csv       480,853 bytes
  review/법인세법/linebreaks_weak.csv   135,033 bytes
  review/법인세법/markers.csv            3,820 bytes
  review/법인세법/merges.csv           194,853 bytes
  review/법인세법/reorder.csv           39,889 bytes
  review/법인세법/result.md            422,766 bytes
  review/법인세법/splits.csv               157 bytes
  review/법인세법/summary.md             1,243 bytes
  review/법인세법/unmatched.csv         57,202 bytes
  review/부가가치세법/elements.csv       155,632 bytes
  review/부가가치세법/linebreaks.csv     163,831 bytes
  review/부가가치세법/linebreaks_weak.csv    45,335 bytes
  review/부가가치세법/markers.csv            325 bytes
  review/부가가치세법/merges.csv          64,306 bytes
  review/부가가치세법/reorder.csv         19,482 bytes
  review/부가가치세법/result.md          147,683 bytes
  review/부가가치세법/

---
## 결과 해석 가이드

| 평가 영역 | 무엇을 보고 판단하나 | 좋은 신호 |
|---|---|---|
| **1. Layout Analysis** | §4의 페이지×요소 교차표, `layout_result.csv` | 본문이 `Text`/`List`, 소제목이 `Heading`, 표가 `Table`로 잡히고, 페이지별 분포가 원본 PDF와 일치 |
| **2. Text Hierarchy** | §5의 계층 트리, Heading 레벨 분포, Level skips | 장/조 구조가 H1/H2로 분리되고 Level skips = 0, 본문이 올바른 상위 Heading 아래에 붙음 |
| **3. Table Structure** | §6의 Rows/Cols/Header/Merged, Markdown·DataFrame | 행·열 수가 원본과 일치, 헤더 행 감지, 병합 셀이 span으로 표현, 셀 어긋남 없음 |

주의해서 볼 실패 패턴
- 머리말/꼬리말이 본문 `Text`로 섞여 들어오는 경우(페이지 경계에서 문장이 끊김)
- 다단·박스 레이아웃에서 읽기 순서(`Order`)가 뒤섞이는 경우
- 표 헤더가 `Not detected`거나 `DF Shape`의 행 수가 `Rows`-1과 맞지 않는 경우(헤더 승격 실패)